# DS-05 — Giải thích mô hình và phân tích lỗi

## Mục tiêu

DS-05 tập trung vào việc giải thích **mô hình được chọn từ DS-04** và phân tích các trường hợp mô hình dự đoán sai.

Nội dung gồm:

1. Feature importance phù hợp với đặc tính của từng mô hình.
2. Phân tích FP/FN và các trường hợp dự đoán sai.
3. Chọn case theo tiêu chí rõ ràng.
4. Giải thích **từng case bằng SHAP**: thực tế là gì, model đoán gì, feature nào đẩy prediction và vì sao prediction đó sai.
5. Kiểm tra độ tin cậy của feature importance bằng **Permutation Importance trên test**.
6. Kiểm tra độ nhạy của Delay Risk khi loại hai historic-rate feature nghi ngờ là nhiễu.

### Quy ước quan trọng

- Với **Delay Risk**, Random Forest là model được giữ lại để explainability theo DS-04.
- Với **Win Probability**, chọn **CatBoost** vì bài toán vận hành là xếp hạng Top-K; CatBoost dẫn **Top-30% Precision = 0.636**, cao hơn Random Forest = 0.545.
- Không trình bày Top-5 feature importance cho Win Probability vì Test ROC-AUC chỉ khoảng 0.513–0.527, cho thấy khả năng phân biệt rất yếu; importance trong trường hợp này không đủ đáng tin để diễn giải thành tín hiệu nghiệp vụ.
- Các kết quả explainability là **giải thích mô hình**, không phải quan hệ nhân quả.

In [10]:
# 0. Nạp thư viện và thiết lập khả năng tái lập
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score

from catboost import CatBoostClassifier
import shap

RANDOM_STATE = 42
BASE_DIR = Path(".")

print("Thư mục làm việc:", Path.cwd())

Thư mục làm việc: d:\Prj_TT_Hidecc\DS05


In [11]:
# 1. Nạp đúng dữ liệu và bộ feature của DS-04
delay_features = [
    "contract_value",
    "planned_duration_days",
    "n_disciplines_known_at_snapshot",
    "n_open_disciplines_at_snapshot",
    "total_workload_pct",
    "pct_staff_overloaded_at_snapshot",
    "n_open_overdue_tasks_at_snapshot",
    "pm_historic_late_rate",
    "customer_historic_late_rate",
    "project_type",
    "discipline",
]

win_features = [
    "lead_source",
    "customer_segment",
    "territory",
    "opportunity_amount",
    "days_lead_to_opp",
    "discount_rate_at_first_quotation",
]

delay_train = pd.read_csv(BASE_DIR / "delay_risk_train.csv")
delay_test = pd.read_csv(BASE_DIR / "delay_risk_test.csv")
win_train = pd.read_csv(BASE_DIR / "win_probability_train.csv")
win_test = pd.read_csv(BASE_DIR / "win_probability_test.csv")

print("Kích thước Delay train/test:", delay_train.shape, delay_test.shape)
print("Kích thước Win train/test:", win_train.shape, win_test.shape)

print("\nSố Opportunity train thực tế:", len(win_train))
assert len(win_train) == 84, "Số Opportunity train không khớp DS-04."
print("Kiểm tra số dòng Win train: OK")

Kích thước Delay train/test: (391, 17) (153, 17)
Kích thước Win train/test: (84, 12) (35, 12)

Số Opportunity train thực tế: 84
Kiểm tra số dòng Win train: OK


## 2. Kiểm tra dữ liệu trước khi giải thích

Cell tiếp theo kiểm tra feature dictionary và các điều kiện tối thiểu trước khi fit lại model.

Điểm quan trọng là **không được hard-code số lượng dữ liệu trong phần diễn giải**. Số dòng phải được lấy trực tiếp từ dataset.

Đối với Delay, `assignee_id` không nằm trong feature dictionary và `discipline` phải được giữ lại. Đây là hai quyết định feature đã được chốt từ DS-04.

In [12]:
# 2. Kiểm tra dữ liệu và feature
assert set(delay_features).issubset(delay_train.columns)
assert set(delay_features).issubset(delay_test.columns)
assert set(win_features).issubset(win_train.columns)
assert set(win_features).issubset(win_test.columns)

assert "assignee_id" not in delay_features
assert "discipline" in delay_features

for df in [delay_train, delay_test, win_train, win_test]:
    assert df["label"].notna().all()

print("Feature Delay:", delay_features)
print("Feature Win:", win_features)
print("Kiểm tra dữ liệu và feature: OK")

Feature Delay: ['contract_value', 'planned_duration_days', 'n_disciplines_known_at_snapshot', 'n_open_disciplines_at_snapshot', 'total_workload_pct', 'pct_staff_overloaded_at_snapshot', 'n_open_overdue_tasks_at_snapshot', 'pm_historic_late_rate', 'customer_historic_late_rate', 'project_type', 'discipline']
Feature Win: ['lead_source', 'customer_segment', 'territory', 'opportunity_amount', 'days_lead_to_opp', 'discount_rate_at_first_quotation']
Kiểm tra dữ liệu và feature: OK


# 3. Delay Risk — Random Forest

Random Forest là mô hình được phân tích chính cho Delay Risk vì trong DS-04 nó đạt Test ROC-AUC cao nhất trong ba baseline.

Mục tiêu của phần này không chỉ là lấy importance, mà còn kiểm tra xem các feature đứng đầu có thực sự có sức dự báo hay chỉ được ưu tiên do đặc tính của **MDI (`feature_importances_`)**.

Do đó sẽ thực hiện hai phép kiểm:

1. **Permutation Importance trên tập test**.
2. **Ablation:** bỏ `customer_historic_late_rate` và `pm_historic_late_rate`, huấn luyện lại và đo Test ROC-AUC.

Đây là hai phép kiểm độc lập với MDI.

In [13]:
# 3.1 Dựng và huấn luyện Random Forest của Delay Risk
def make_rf_pipeline(train_df, features):
    X = train_df[features]
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = [c for c in features if c not in numeric]

    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])

    model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])

delay_model = make_rf_pipeline(delay_train, delay_features)

X_delay_train = delay_train[delay_features].copy()
y_delay_train = delay_train["label"].astype(int)
X_delay_test = delay_test[delay_features].copy()
y_delay_test = delay_test["label"].astype(int)

delay_model.fit(X_delay_train, y_delay_train)
delay_prob = delay_model.predict_proba(X_delay_test)[:, 1]
delay_pred = (delay_prob >= 0.5).astype(int)

delay_auc = roc_auc_score(y_delay_test, delay_prob)

print(f"ROC-AUC trên tập test của Delay: {delay_auc:.3f}")
print("Ma trận nhầm lẫn của Delay:")
print(confusion_matrix(y_delay_test, delay_pred, labels=[0, 1]))

ROC-AUC trên tập test của Delay: 0.640
Ma trận nhầm lẫn của Delay:
[[48 29]
 [30 46]]


### 3.2. MDI `feature_importances_`: chỉ dùng làm tham khảo

Random Forest có `feature_importances_`, còn gọi là **Mean Decrease in Impurity (MDI)**. Chỉ số này có thể ưu tiên các feature liên tục có nhiều giá trị khác nhau.

Vì vậy Top-5 MDI **không được tự động xem là Top-5 feature có tín hiệu thật**.

Đặc biệt, hai feature:

- `customer_historic_late_rate`
- `pm_historic_late_rate`

có rất nhiều giá trị khác nhau trên tập train. Trong khi đó, theo thông tin từ quá trình sinh dữ liệu, độ trễ được gieo theo từng project và **độc lập với khách hàng và PM**. Vì vậy cần kiểm tra hai feature này bằng phép đo khác trước khi diễn giải chúng.

In [14]:
# 3.2 Tính MDI và gộp các cột one-hot về feature gốc
def aggregate_mdi_importance(pipe, features):
    prep = pipe.named_steps["prep"]
    model = pipe.named_steps["model"]
    names = prep.get_feature_names_out()
    values = model.feature_importances_

    rows = []
    for name, value in zip(names, values):
        raw = name.split("__", 1)[1] if "__" in name else name
        base = next(
            (f for f in features if raw == f or raw.startswith(f + "_")),
            raw,
        )
        rows.append((base, float(value)))

    out = pd.DataFrame(rows, columns=["feature", "importance"])
    out = out.groupby("feature", as_index=False)["importance"].sum()
    out["importance_pct"] = out["importance"] / out["importance"].sum() * 100
    return out.sort_values("importance", ascending=False).reset_index(drop=True)

delay_mdi = aggregate_mdi_importance(delay_model, delay_features)

print("MDI — Top-5 feature của Delay Risk")
display(delay_mdi.head(5))

MDI — Top-5 feature của Delay Risk


,feature,importance,importance_pct
0,customer_historic_late_rate,0.191830,19.183035
1,pm_historic_late_rate,0.158450,15.845016
2,pct_staff_overloaded_at_snapshot,0.101074,10.107437
3,planned_duration_days,0.090276,9.027639
4,discipline,0.085654,8.565423


### 3.3. Permutation Importance trên tập test

Permutation Importance đo mức giảm hiệu năng khi xáo trộn một feature trên **tập test**.

Khác với MDI, phép đo này trực tiếp hỏi:

> Nếu phá vỡ thông tin của feature này, Test ROC-AUC giảm bao nhiêu?

Vì vậy đây là phép kiểm quan trọng để xác định Top-5 MDI có thực sự mang tín hiệu dự báo hay không.

Nếu importance gần 0 hoặc âm, không nên coi feature đó là tín hiệu đáng tin chỉ vì MDI xếp nó cao.

**Kết quả nói lên điều gì?**

Permutation Importance không chứng minh feature có quan hệ nhân quả. Nó chỉ cho biết model đang phụ thuộc vào feature đó đến mức nào trên tập test.

Nếu một feature có giá trị permutation dương, việc phá vỡ feature đó làm AUC giảm trên test. Nếu giá trị gần 0 hoặc âm, feature không cho thấy đóng góp dự báo ổn định trong phép kiểm này.

In [15]:
# 3.3 Permutation Importance trên tập test
perm = permutation_importance(
    delay_model,
    X_delay_test,
    y_delay_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

delay_perm = pd.DataFrame({
    "feature": X_delay_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("Permutation Importance trên tập test — Top-5")
display(delay_perm.head(5))

Permutation Importance trên tập test — Top-5


,feature,importance_mean,importance_std
0,customer_historic_late_rate,0.043598,0.027531
1,n_disciplines_known_at_snapshot,0.028355,0.012252
2,pm_historic_late_rate,0.024960,0.011660
3,discipline,0.018563,0.007754
4,pct_staff_overloaded_at_snapshot,0.015715,0.011080


### 3.4. Kiểm tra ablation của hai historic-rate feature

Phép kiểm thứ hai là bỏ hoàn toàn:

- `customer_historic_late_rate`
- `pm_historic_late_rate`

sau đó huấn luyện lại Random Forest với các feature còn lại.

Câu hỏi cần trả lời:

> **Nếu hai feature này không mang tín hiệu thực sự, việc bỏ chúng có làm Test ROC-AUC giảm đáng kể không?**

Kết quả này phải được so trực tiếp với ROC-AUC của model đầy đủ.

**Kết quả nói lên điều gì?**

- AUC giảm sau ablation → model hiện tại đang khai thác thông tin từ hai feature.
- AUC gần như không đổi → hai feature không cần thiết cho hiệu năng của model.
- AUC tăng → hai feature có thể đang làm model học nhiễu.

Trong trường hợp của dữ liệu này, kết quả phải được đọc cùng với thông tin thiết kế rằng độ trễ được gieo độc lập với customer và PM.

In [16]:
# 3.4 Ablation: bỏ hai historic-rate feature
historic_features = [
    "customer_historic_late_rate",
    "pm_historic_late_rate",
]

delay_features_without_history = [
    f for f in delay_features if f not in historic_features
]

delay_model_without_history = make_rf_pipeline(
    delay_train,
    delay_features_without_history,
)

delay_model_without_history.fit(
    delay_train[delay_features_without_history],
    y_delay_train,
)

delay_prob_without_history = delay_model_without_history.predict_proba(
    delay_test[delay_features_without_history]
)[:, 1]

delay_auc_without_history = roc_auc_score(
    y_delay_test,
    delay_prob_without_history,
)

auc_drop = delay_auc - delay_auc_without_history

print(f"ROC-AUC đầy đủ: {delay_auc:.3f}")
print(f"ROC-AUC bỏ hai historic-rate: {delay_auc_without_history:.3f}")
print(f"Mức thay đổi ROC-AUC: {auc_drop:+.3f}")

ROC-AUC đầy đủ: 0.640
ROC-AUC bỏ hai historic-rate: 0.597
Mức thay đổi ROC-AUC: +0.043


### 3.5. Kết luận Top-5 Delay

Không lấy `feature_importances_` (MDI) làm Top-5 cuối cùng vì MDI có thể thiên vị feature liên tục nhiều giá trị.

Hai phép kiểm được dùng:

1. **Permutation Importance trên test** để đo mức suy giảm Test ROC-AUC khi xáo trộn feature.
2. **Ablation** để đo Test ROC-AUC sau khi bỏ `customer_historic_late_rate` và `pm_historic_late_rate`.

Kết quả được đọc như sau:

- Nếu MDI cao nhưng permutation gần 0/âm, feature đó không có bằng chứng dự báo ổn định.
- Nếu permutation dương và ablation làm AUC giảm, model thực sự đang sử dụng feature đó trong mẫu hiện tại.
- Tuy nhiên, theo thiết kế dữ liệu, độ trễ được gieo độc lập với customer và PM. Vì vậy hai historic-rate feature không được xem là **tín hiệu nghiệp vụ đáng tin**, dù chúng có thể giúp model trên mẫu hiện tại.
- Do đó Top-5 dùng để **diễn giải nghiệp vụ** được lấy từ Permutation Importance sau khi loại hai historic-rate feature nghi ngờ.

Điểm quan trọng là không che giấu kết quả ablation: việc bỏ hai feature làm AUC thay đổi bao nhiêu vẫn phải được báo cáo rõ ràng.

In [17]:
# 3.5 Bảng tổng hợp hai phép kiểm cho hai historic-rate feature
history_check = delay_perm[
    delay_perm["feature"].isin(historic_features)
].copy()

history_check["MDI_importance_pct"] = history_check["feature"].map(
    delay_mdi.set_index("feature")["importance_pct"]
)

display(
    history_check[
        ["feature", "MDI_importance_pct", "importance_mean", "importance_std"]
    ].sort_values("MDI_importance_pct", ascending=False)
)

print(
    f"Chênh lệch ROC-AUC sau ablation hai historic-rate: "
    f"{delay_auc_without_history - delay_auc:+.3f}"
)

# Top-5 dùng để diễn giải cuối cùng: loại hai historic-rate feature khỏi
# bảng xếp hạng nghiệp vụ vì chúng không có tín hiệu được thiết kế sẵn.
delay_perm_interpretable = delay_perm[
    ~delay_perm["feature"].isin(historic_features)
].copy()

print("Top-5 feature Delay dùng cho diễn giải sau khi loại hai historic-rate:")
display(delay_perm_interpretable.head(5))

print(
    "\nKết luận kiểm tra: hai historic-rate có thể vẫn giúp RF trên mẫu test hiện tại,"
    "\nnhưng do dữ liệu được sinh với độ trễ độc lập với customer/PM, "
    "\nkhông sử dụng chúng làm insight nghiệp vụ. "
    f"\nViệc bỏ cả hai làm ROC-AUC thay đổi {delay_auc_without_history - delay_auc:+.3f}."
)


,feature,MDI_importance_pct,importance_mean,importance_std
0,customer_historic_late_rate,19.183035,0.043598,0.027531
2,pm_historic_late_rate,15.845016,0.024960,0.011660


Chênh lệch ROC-AUC sau ablation hai historic-rate: -0.043
Top-5 feature Delay dùng cho diễn giải sau khi loại hai historic-rate:


,feature,importance_mean,importance_std
1,n_disciplines_known_at_snapshot,0.028355,0.012252
3,discipline,0.018563,0.007754
4,pct_staff_overloaded_at_snapshot,0.015715,0.011080
5,n_open_overdue_tasks_at_snapshot,0.008162,0.004671
6,project_type,0.007809,0.010968



Kết luận kiểm tra: hai historic-rate có thể vẫn giúp RF trên mẫu test hiện tại,
nhưng do dữ liệu được sinh với độ trễ độc lập với customer/PM, 
không sử dụng chúng làm insight nghiệp vụ. 
Việc bỏ cả hai làm ROC-AUC thay đổi -0.043.


In [18]:
# 3.5.x Tạo bảng lỗi FP/FN cho Delay Risk
# Dùng toàn bộ delay_test để giữ lại project_id/discipline cho error analysis.
# source_index dùng để truy ngược đúng observation gốc khi tính SHAP.

delay_error = delay_test.copy()

# delay_prob / delay_pred được tạo ở cell 3.1 và cùng thứ tự với delay_test.
assert len(delay_error) == len(delay_prob) == len(delay_pred)

delay_error["predicted_probability"] = np.asarray(delay_prob)
delay_error["predicted_label"] = np.asarray(delay_pred).astype(int)
delay_error["source_index"] = delay_error.index

delay_error["error_type"] = np.select(
    [
        (delay_error["label"].astype(int) == 0) & (delay_error["predicted_label"] == 1),
        (delay_error["label"].astype(int) == 1) & (delay_error["predicted_label"] == 0),
    ],
    ["FP", "FN"],
    default="Correct",
)

delay_error["distance_from_threshold"] = (
    delay_error["predicted_probability"] - 0.5
).abs()

print("Delay error analysis:")
display(delay_error["error_type"].value_counts())

display(
    delay_error[
        [
            "project_id",
            "discipline",
            "label",
            "predicted_label",
            "predicted_probability",
            "error_type",
            "distance_from_threshold",
        ]
    ].head()
)


Delay error analysis:


error_type
Correct    94
FN         30
FP         29
Name: count, dtype: int64

,project_id,discipline,label,predicted_label,predicted_probability,error_type,distance_from_threshold
0,PROJ-0006,Structure,0,1,0.587272,FP,0.087272
1,PROJ-0006,Fire Protection,1,0,0.496612,FN,0.003388
2,PROJ-0006,ELV,0,1,0.653246,FP,0.153246
3,PROJ-0006,Architecture,0,1,0.561298,FP,0.061298
4,PROJ-0006,HVAC,1,1,0.700176,Correct,0.200176


### 3.6. Phân tích FP/FN

Confusion matrix cho biết mô hình sai theo hai hướng:

- **FP:** model cảnh báo trễ nhưng thực tế không trễ.
- **FN:** model dự đoán không trễ nhưng thực tế lại trễ.

Sau khi xác định FP/FN, ta chọn năm case tiêu biểu:

1. FP có xác suất trễ cao nhất.
2. FN có xác suất trễ thấp nhất.
3. Error gần threshold 0.5 nhất.

Mục đích là phân tích cả lỗi **cảnh báo quá mức**, **bỏ sót rủi ro** và **trường hợp model không chắc chắn**.

In [19]:
# 3.6 Tạo bảng FP/FN và chọn 3 case
# 2 FP
delay_fp = (
    delay_error[delay_error["error_type"] == "FP"]
    .sort_values("predicted_probability", ascending=False)
    .head(2)
)

# 2 FN
delay_fn = (
    delay_error[delay_error["error_type"] == "FN"]
    .sort_values("predicted_probability", ascending=True)
    .head(2)
)

# Các case đã chọn
selected_ids = set(
    zip(delay_fp["project_id"], delay_fp["discipline"])
) | set(
    zip(delay_fn["project_id"], delay_fn["discipline"])
)

# Tìm borderline trong các case chưa được chọn
borderline_candidates = delay_error[
    delay_error["error_type"].isin(["FP", "FN"])
].copy()

borderline_candidates["case_id"] = list(
    zip(
        borderline_candidates["project_id"],
        borderline_candidates["discipline"]
    )
)

delay_borderline = (
    borderline_candidates[
        ~borderline_candidates["case_id"].isin(selected_ids)
    ]
    .sort_values("distance_from_threshold", ascending=True)
    .head(1)
)

# Gộp
delay_cases = pd.concat(
    [delay_fp, delay_fn, delay_borderline],
    ignore_index=True,
)

print("\nNăm case Delay được chọn:")
display(
    delay_cases[
        [
            "project_id",
            "discipline",
            "label",
            "predicted_label",
            "predicted_probability",
            "error_type",
            "distance_from_threshold",
        ]
    ]
)


Năm case Delay được chọn:


,project_id,discipline,label,predicted_label,predicted_probability,error_type,distance_from_threshold
0,PROJ-0029,HVAC,0,1,0.799642,FP,0.299642
1,PROJ-0028,Cost Estimate,0,1,0.790175,FP,0.290175
2,PROJ-0026,MEP,1,0,0.182969,FN,0.317031
3,PROJ-0059,ELV,1,0,0.204389,FN,0.295611
4,PROJ-0018,Site Master Plan,0,1,0.503090,FP,0.003090


### 3.7. Giải thích trực tiếp từng case Delay bằng SHAP

Bảng SHAP chỉ là dữ liệu trung gian. **Phần bắt buộc của DS-05 là đọc và giải thích case.**

Với mỗi case, phần dưới sẽ in:

- thực tế là gì;
- model dự đoán gì;
- xác suất model;
- Top-5 feature có |SHAP| lớn nhất;
- feature nào đẩy prediction về phía "trễ";
- feature nào kéo prediction ngược lại.

Sau đó sinh một đoạn diễn giải bằng Markdown để trả lời:

> **Vì sao model dự đoán như vậy và vì sao prediction đó sai?**

In [20]:
from IPython.display import display, Markdown


# ============================================================
# 3.7 Local SHAP explanation cho 5 error cases
# ============================================================

def local_shap_table(pipe, X, features, row_index):
    """
    Tính SHAP cho một observation và gom SHAP về feature gốc.

    Lưu ý:
    - Với pipeline có preprocessing (imputation + scaling + one-hot),
      SHAP được tính trên feature sau preprocessing.
    - Các dummy variables được gom ngược về feature gốc.
    """
    prep = pipe.named_steps["prep"]
    model = pipe.named_steps["model"]

    # Transform dữ liệu giống lúc model prediction
    Xt = prep.transform(X[features])

    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()

    # TreeExplainer cho Random Forest
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(Xt)

    # Xử lý output SHAP tùy version SHAP
    if isinstance(values, list):
        # Binary classification:
        # class 1 = TRỄ
        row_values = np.asarray(values[1])[row_index]
    else:
        arr = np.asarray(values)

        if arr.ndim == 5:
            # Một số version SHAP có shape đặc biệt
            row_values = arr[row_index, :, 1]
        elif arr.ndim == 3:
            # shape: (n_samples, n_features, n_classes)
            row_values = arr[row_index, :, 1]
        else:
            row_values = arr[row_index]

    # Tên feature sau preprocessing
    names = prep.get_feature_names_out()

    rows = []

    for name, value in zip(names, row_values):

        # Ví dụ:
        # num__contract_value
        # cat__discipline_HVAC
        raw = name.split("__", 1)[1] if "__" in name else name

        # Gom dummy variable về feature gốc
        base = next(
            (
                feature
                for feature in features
                if raw == feature or raw.startswith(feature + "_")
            ),
            raw,
        )

        rows.append(
            (
                base,
                float(value)
            )
        )

    out = pd.DataFrame(
        rows,
        columns=["feature", "shap_value"]
    )

    # Một categorical feature có thể tạo ra nhiều dummy variables
    # nên cần cộng SHAP của các dummy về feature gốc
    out = (
        out
        .groupby("feature", as_index=False)["shap_value"]
        .sum()
    )

    out["abs_shap"] = out["shap_value"].abs()

    return (
        out
        .sort_values("abs_shap", ascending=False)
        .reset_index(drop=True)
    )


def delay_case_markdown(case, shap_table):
    """
    Sinh phần diễn giải Markdown cho một error case.
    """

    # Lấy Top-5 feature có ảnh hưởng lớn nhất về độ lớn SHAP
    top = shap_table.head(5)

    positive = (
        top[top["shap_value"] > 0]
        .sort_values("shap_value", ascending=False)
    )

    negative = (
        top[top["shap_value"] < 0]
        .sort_values("shap_value", ascending=True)
    )

    # --------------------------------------------------------
    # Actual / Prediction
    # --------------------------------------------------------

    actual_text = (
        "TRỄ"
        if int(case["label"]) == 1
        else "KHÔNG TRỄ"
    )

    pred_text = (
        "TRỄ"
        if int(case["predicted_label"]) == 1
        else "KHÔNG TRỄ"
    )

    # --------------------------------------------------------
    # Case title
    # --------------------------------------------------------

    project_id = case["project_id"]

    discipline = (
        case["discipline"]
        if "discipline" in case.index
        and pd.notna(case["discipline"])
        else "N/A"
    )

    error_type = case["error_type"]

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    lines = [
        f"### Case `{project_id}` — `{discipline}` — {error_type}",
        "",
        f"- **Thực tế:** {actual_text}.",
        (
            f"- **Model dự đoán:** {pred_text}, "
            f"với xác suất trễ "
            f"**{case['predicted_probability']:.3f}**."
        ),
        f"- **Loại lỗi:** `{error_type}`.",
        "",
        (
            "**Các feature có đóng góp lớn nhất vào prediction "
            "theo SHAP:**"
        ),
    ]

    # --------------------------------------------------------
    # Positive SHAP
    # --------------------------------------------------------

    lines += [
        "",
        "**Feature đẩy prediction về phía TRỄ:**"
    ]

    if len(positive) > 0:

        for r in positive.itertuples():

            lines.append(
                f"- `{r.feature}`: "
                f"SHAP = **{r.shap_value:+.4f}**"
            )

    else:

        lines.append(
            "- Không có feature trong Top-5 "
            "có SHAP dương."
        )

    # --------------------------------------------------------
    # Negative SHAP
    # --------------------------------------------------------

    lines += [
        "",
        "**Feature kéo prediction khỏi phía TRỄ:**"
    ]

    if len(negative) > 0:

        for r in negative.itertuples():

            lines.append(
                f"- `{r.feature}`: "
                f"SHAP = **{r.shap_value:+.4f}**"
            )

    else:

        lines.append(
            "- Không có feature trong Top-5 "
            "có SHAP âm."
        )

    # --------------------------------------------------------
    # Interpretation
    # --------------------------------------------------------

    if error_type == "FP":

        interpretation = (
            "Đây là trường hợp **False Positive (FP)**: "
            "model cảnh báo trễ nhưng thực tế không trễ. "
            "Các feature có SHAP dương là những tín hiệu "
            "khiến xác suất dự đoán nghiêng về phía TRỄ; "
            "tuy nhiên, các tín hiệu này không đủ để model "
            "phân loại đúng outcome thực tế của case."
        )

    elif error_type == "FN":

        interpretation = (
            "Đây là trường hợp **False Negative (FN)**: "
            "model đánh giá rủi ro trễ thấp nhưng thực tế bị trễ. "
            "Các feature có SHAP âm là những tín hiệu "
            "kéo prediction khỏi phía TRỄ, góp phần khiến "
            "model bỏ sót case này."
        )

    else:

        interpretation = (
            "Đây là trường hợp **borderline error**: "
            "xác suất dự đoán nằm rất gần threshold 0.5. "
            "Do đó model không có mức độ chắc chắn cao "
            "trong việc phân loại case này."
        )

    lines += [
        "",
        f"> **Diễn giải:** {interpretation}",
        "",
        (
            "> **Lưu ý:** SHAP mô tả đóng góp của feature "
            "đối với prediction của model, không chứng minh "
            "quan hệ nhân quả giữa feature và việc dự án bị trễ."
        )
    ]

    return "\n".join(lines)


# ============================================================
# Hiển thị 5 error cases
# ============================================================

print("=" * 80)
print("LOCAL SHAP EXPLANATION — 5 DELAY ERROR CASES")
print("=" * 80)


for case_no, (_, case) in enumerate(
    delay_cases.iterrows(),
    start=1
):

    # --------------------------------------------------------
    # Tính SHAP
    # --------------------------------------------------------

    # delay_cases dùng ignore_index nên case.name không còn là index gốc.
    # Dùng source_index để truy đúng observation trong delay_test.
    row_index = delay_test.index.get_loc(case["source_index"])

    shap_table = local_shap_table(
        delay_model,
        delay_test,
        delay_features,
        row_index,
    )

    # --------------------------------------------------------
    # Thông tin case
    # --------------------------------------------------------

    actual_text = (
        "Trễ"
        if int(case["label"]) == 1
        else "Không trễ"
    )

    predicted_text = (
        "Trễ"
        if int(case["predicted_label"]) == 1
        else "Không trễ"
    )

    discipline = (
        case["discipline"]
        if "discipline" in case.index
        and pd.notna(case["discipline"])
        else "N/A"
    )

    print(
        f"\n{'-' * 80}\n"
        f"CASE {case_no}/5 | "
        f"Project={case['project_id']} | "
        f"Discipline={discipline} | "
        f"Error={case['error_type']}\n"
        f"Thực tế={actual_text} | "
        f"Dự đoán={predicted_text} | "
        f"P(trễ)={case['predicted_probability']:.3f}"
    )

    # --------------------------------------------------------
    # Top-5 SHAP table
    # --------------------------------------------------------

    display(
        shap_table[
            ["feature", "shap_value", "abs_shap"]
        ].head(5)
    )

    # --------------------------------------------------------
    # Markdown interpretation
    # --------------------------------------------------------

    display(
        Markdown(
            delay_case_markdown(
                case,
                shap_table
            )
        )
    )

LOCAL SHAP EXPLANATION — 5 DELAY ERROR CASES

--------------------------------------------------------------------------------
CASE 1/5 | Project=PROJ-0029 | Discipline=HVAC | Error=FP
Thực tế=Không trễ | Dự đoán=Trễ | P(trễ)=0.800


,feature,shap_value,abs_shap
0,customer_historic_late_rate,0.095539,0.095539
1,pm_historic_late_rate,0.046737,0.046737
2,project_type,0.039902,0.039902
3,planned_duration_days,0.035679,0.035679
4,pct_staff_overloaded_at_snapshot,0.029705,0.029705


### Case `PROJ-0029` — `HVAC` — FP

- **Thực tế:** KHÔNG TRỄ.
- **Model dự đoán:** TRỄ, với xác suất trễ **0.800**.
- **Loại lỗi:** `FP`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy prediction về phía TRỄ:**
- `customer_historic_late_rate`: SHAP = **+0.0955**
- `pm_historic_late_rate`: SHAP = **+0.0467**
- `project_type`: SHAP = **+0.0399**
- `planned_duration_days`: SHAP = **+0.0357**
- `pct_staff_overloaded_at_snapshot`: SHAP = **+0.0297**

**Feature kéo prediction khỏi phía TRỄ:**
- Không có feature trong Top-5 có SHAP âm.

> **Diễn giải:** Đây là trường hợp **False Positive (FP)**: model cảnh báo trễ nhưng thực tế không trễ. Các feature có SHAP dương là những tín hiệu khiến xác suất dự đoán nghiêng về phía TRỄ; tuy nhiên, các tín hiệu này không đủ để model phân loại đúng outcome thực tế của case.

> **Lưu ý:** SHAP mô tả đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và việc dự án bị trễ.


--------------------------------------------------------------------------------
CASE 2/5 | Project=PROJ-0028 | Discipline=Cost Estimate | Error=FP
Thực tế=Không trễ | Dự đoán=Trễ | P(trễ)=0.790


,feature,shap_value,abs_shap
0,customer_historic_late_rate,0.123744,0.123744
1,n_disciplines_known_at_snapshot,0.061279,0.061279
2,planned_duration_days,0.054495,0.054495
3,pm_historic_late_rate,-0.052126,0.052126
4,discipline,0.030482,0.030482


...

,feature,shap_value,abs_shap
0,project_type,-0.057532,0.057532
1,customer_historic_late_rate,-0.055770,0.055770
2,pm_historic_late_rate,-0.045093,0.045093
3,pct_staff_overloaded_at_snapshot,-0.038434,0.038434
4,contract_value,-0.033361,0.033361


### Case `PROJ-0059` — `ELV` — FN

- **Thực tế:** TRỄ.
- **Model dự đoán:** KHÔNG TRỄ, với xác suất trễ **0.204**.
- **Loại lỗi:** `FN`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy prediction về phía TRỄ:**
- Không có feature trong Top-5 có SHAP dương.

**Feature kéo prediction khỏi phía TRỄ:**
- `project_type`: SHAP = **-0.0575**
- `customer_historic_late_rate`: SHAP = **-0.0558**
- `pm_historic_late_rate`: SHAP = **-0.0451**
- `pct_staff_overloaded_at_snapshot`: SHAP = **-0.0384**
- `contract_value`: SHAP = **-0.0334**

> **Diễn giải:** Đây là trường hợp **False Negative (FN)**: model đánh giá rủi ro trễ thấp nhưng thực tế bị trễ. Các feature có SHAP âm là những tín hiệu kéo prediction khỏi phía TRỄ, góp phần khiến model bỏ sót case này.

> **Lưu ý:** SHAP mô tả đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và việc dự án bị trễ.


--------------------------------------------------------------------------------
CASE 5/5 | Project=PROJ-0018 | Discipline=Site Master Plan | Error=FP
Thực tế=Không trễ | Dự đoán=Trễ | P(trễ)=0.503


,feature,shap_value,abs_shap
0,customer_historic_late_rate,-0.061161,0.061161
1,project_type,0.043189,0.043189
2,total_workload_pct,0.028432,0.028432
3,n_open_disciplines_at_snapshot,0.017113,0.017113
4,contract_value,-0.013915,0.013915


### Case `PROJ-0018` — `Site Master Plan` — FP

- **Thực tế:** KHÔNG TRỄ.
- **Model dự đoán:** TRỄ, với xác suất trễ **0.503**.
- **Loại lỗi:** `FP`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy prediction về phía TRỄ:**
- `project_type`: SHAP = **+0.0432**
- `total_workload_pct`: SHAP = **+0.0284**
- `n_open_disciplines_at_snapshot`: SHAP = **+0.0171**

**Feature kéo prediction khỏi phía TRỄ:**
- `customer_historic_late_rate`: SHAP = **-0.0612**
- `contract_value`: SHAP = **-0.0139**

> **Diễn giải:** Đây là trường hợp **False Positive (FP)**: model cảnh báo trễ nhưng thực tế không trễ. Các feature có SHAP dương là những tín hiệu khiến xác suất dự đoán nghiêng về phía TRỄ; tuy nhiên, các tín hiệu này không đủ để model phân loại đúng outcome thực tế của case.

> **Lưu ý:** SHAP mô tả đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và việc dự án bị trễ.

# 4. Win Probability — CatBoost

Theo yêu cầu của bài toán, Win Probability là bài toán **xếp hạng để ưu tiên cơ hội**, không phải chỉ phân loại Won/Lost tại một threshold.

Vì vậy metric vận hành chính là **Top-30% Precision**. Trong DS-04:

- CatBoost: **0.636**
- Random Forest: **0.545**
- Logistic Regression: **0.455**

Do đó DS-05 chọn **CatBoost** làm model chính để giải thích Win Probability.

Mặc dù Test ROC-AUC của CatBoost chỉ **0.513**, đây là dấu hiệu model gần như không phân biệt được trên test. Vì vậy **không trình bày Top-5 feature importance của Win** và không cố tối ưu AUC chỉ để làm metric đẹp.

Thay vào đó, DS-05 tập trung vào các case xếp hạng sai để minh họa giới hạn của model.

In [21]:
# 4.1 Dựng CatBoost native categorical đúng tinh thần DS-04
cat_features_win = ["lead_source", "customer_segment", "territory"]

win_model = CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False,
    random_seed=RANDOM_STATE,
    cat_features=cat_features_win,
)

X_win_train = win_train[win_features].copy()
y_win_train = win_train["label"].astype(int)
X_win_test = win_test[win_features].copy()
y_win_test = win_test["label"].astype(int)

win_model.fit(X_win_train, y_win_train)

win_prob = win_model.predict_proba(X_win_test)[:, 1]
win_pred = (win_prob >= 0.5).astype(int)

win_auc = roc_auc_score(y_win_test, win_prob)

print(f"ROC-AUC trên tập test của CatBoost Win: {win_auc:.3f}")
print("Số Opportunity train thực tế:", len(win_train))
print("Số Opportunity test thực tế:", len(win_test))

ROC-AUC trên tập test của CatBoost Win: 0.497
Số Opportunity train thực tế: 84
Số Opportunity test thực tế: 35


### 4.2. Vì sao không trình bày Top-5 feature importance của Win?

Test ROC-AUC của CatBoost chỉ khoảng **0.513**, trong khi 0.5 là mức tương đương khả năng phân biệt ngẫu nhiên.

Do đó, một bảng feature importance đẹp về mặt hình thức có thể tạo ra cảm giác rằng model có tín hiệu rõ ràng trong khi Test ROC-AUC không ủng hộ kết luận đó.

Vì vậy DS-05 **không dùng Top-5 importance của Win để diễn giải nghiệp vụ**.

Đây là một quyết định có chủ đích: **không cố biến nhiễu thành insight**.

### 4.3. Phân tích các case xếp hạng sai của Win

Để vẫn thực hiện yêu cầu giải thích lỗi, chọn 3 case:

1. **High-confidence FP:** probability cao nhưng thực tế Lost.
2. **High-confidence FN:** probability thấp nhưng thực tế Won.
3. **Borderline error:** probability gần 0.5 nhất.

Do Win là bài toán xếp hạng, FP/FN tại threshold 0.5 chỉ là **chẩn đoán**, không phải metric vận hành chính.

In [22]:
# ============================================================
# 4.3 Tạo bảng lỗi và chọn 5 case Win
#     2 FP + 2 FN + 1 borderline
# ============================================================

win_error = win_test.copy()

win_error["predicted_probability"] = win_prob
win_error["predicted_label"] = win_pred

win_error["error_type"] = np.select(
    [
        (win_error["label"] == 0) & (win_error["predicted_label"] == 1),
        (win_error["label"] == 1) & (win_error["predicted_label"] == 0),
    ],
    ["FP", "FN"],
    default="Đúng",
)

win_error["distance_from_threshold"] = (
    win_error["predicted_probability"] - 0.5
).abs()


# ------------------------------------------------------------
# 2 False Positive
# Model dự đoán WON nhưng thực tế LOST
# Chọn các case có probability cao nhất
# ------------------------------------------------------------

win_fp = (
    win_error[
        win_error["error_type"] == "FP"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(2)
)


# ------------------------------------------------------------
# 2 False Negative
# Model dự đoán LOST nhưng thực tế WON
# Chọn các case có probability thấp nhất
# ------------------------------------------------------------

win_fn = (
    win_error[
        win_error["error_type"] == "FN"
    ]
    .sort_values(
        "predicted_probability",
        ascending=True
    )
    .head(2)
)


# ------------------------------------------------------------
# Loại các case đã được chọn
# để đảm bảo borderline là case khác
# ------------------------------------------------------------

selected_ids = set(
    win_fp["opportunity_id"]
).union(
    set(win_fn["opportunity_id"])
)


# ------------------------------------------------------------
# 1 Borderline error
# Chọn error case có probability gần 0.5 nhất
# nhưng không trùng 4 case phía trên
# ------------------------------------------------------------

win_borderline_candidates = (
    win_error[
        win_error["error_type"].isin(["FP", "FN"])
    ]
    .copy()
)

win_borderline_candidates = (
    win_borderline_candidates[
        ~win_borderline_candidates["opportunity_id"]
        .isin(selected_ids)
    ]
    .sort_values(
        "distance_from_threshold",
        ascending=True
    )
    .head(1)
)


# ------------------------------------------------------------
# Ghép thành 5 case
# ------------------------------------------------------------

win_cases = (
    pd.concat(
        [
            win_fp,
            win_fn,
            win_borderline_candidates,
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["opportunity_id"]
    )
)


# ------------------------------------------------------------
# Kiểm tra số lượng
# ------------------------------------------------------------

print(f"Số case được chọn: {len(win_cases)}")
print("\nPhân loại case:")
display(
    win_cases["error_type"]
    .value_counts()
)


# ------------------------------------------------------------
# Hiển thị danh sách 5 case
# ------------------------------------------------------------

print("\nCác case Win được chọn:")

display(
    win_cases[
        [
            "opportunity_id",
            "label",
            "predicted_label",
            "predicted_probability",
            "error_type",
            "distance_from_threshold",
        ]
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

Số case được chọn: 5

Phân loại case:


error_type
FN    3
FP    2
Name: count, dtype: int64


Các case Win được chọn:


,opportunity_id,label,predicted_label,predicted_probability,error_type,distance_from_threshold
0,CRM-OPP-2026-00148,0,1,0.933535,FP,0.433535
1,CRM-OPP-2026-00037,0,1,0.762861,FP,0.262861
2,CRM-OPP-2026-00034,1,0,0.455128,FN,0.044872
3,CRM-OPP-2026-00007,1,0,0.110052,FN,0.389948
4,CRM-OPP-2026-00009,1,0,0.044830,FN,0.455170


### 4.4. Giải thích từng case Win bằng SHAP

Với Win, SHAP được dùng để trả lời:

> **Tại sao model lại đánh xác suất thắng cao/thấp cho Opportunity này?**

Mỗi case sẽ được trình bày theo:

- thực tế Won/Lost;
- prediction của model;
- probability;
- Top-5 feature theo |SHAP|;
- feature đẩy probability về phía Won;
- feature kéo probability khỏi phía Won;
- lý do prediction bị xem là sai.

Do Test chỉ có 35 Opportunity và AUC gần 0.5, phần này được xem là **diagnostic**, không phải bằng chứng rằng các feature có quan hệ nhân quả với việc thắng thầu.

In [23]:
from IPython.display import display, Markdown


# ============================================================
# 4.4 Local SHAP explanation cho 5 Win error cases
# ============================================================

def catboost_local_shap_table(model, X, features, row_index):
    """
    Tính local SHAP cho một Opportunity.

    Model:
        CatBoostClassifier

    Output:
        SHAP value của từng feature gốc.
    """

    explainer = shap.TreeExplainer(model)

    values = explainer.shap_values(
        X[features]
    )

    # --------------------------------------------------------
    # Xử lý output SHAP tùy version
    # --------------------------------------------------------

    if isinstance(values, list):

        # Binary classification
        # class 1 = WON
        row_values = np.asarray(
            values[1]
        )[row_index]

    else:

        arr = np.asarray(values)

        if arr.ndim == 3:
            # shape:
            # (n_samples, n_features, n_classes)

            row_values = arr[
                row_index,
                :,
                1
            ]

        elif arr.ndim == 2:
            # shape:
            # (n_samples, n_features)

            row_values = arr[
                row_index
            ]

        else:
            row_values = arr[
                row_index
            ]

    # --------------------------------------------------------
    # Tạo bảng SHAP
    # --------------------------------------------------------

    out = pd.DataFrame(
        {
            "feature": features,
            "shap_value": row_values,
        }
    )

    out["abs_shap"] = (
        out["shap_value"].abs()
    )

    return (
        out
        .sort_values(
            "abs_shap",
            ascending=False
        )
        .reset_index(drop=True)
    )


def win_case_markdown(case, shap_table):
    """
    Sinh phần diễn giải Markdown cho một Win error case.
    """

    # Top-5 feature theo absolute SHAP
    top = shap_table.head(5)

    positive = (
        top[
            top["shap_value"] > 0
        ]
        .sort_values(
            "shap_value",
            ascending=False
        )
    )

    negative = (
        top[
            top["shap_value"] < 0
        ]
        .sort_values(
            "shap_value",
            ascending=True
        )
    )

    # --------------------------------------------------------
    # Actual / Prediction
    # --------------------------------------------------------

    actual_text = (
        "WON"
        if int(case["label"]) == 1
        else "LOST"
    )

    pred_text = (
        "WON"
        if int(case["predicted_label"]) == 1
        else "LOST"
    )

    opportunity_id = (
        case["opportunity_id"]
    )

    error_type = (
        case["error_type"]
    )

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    lines = [
        f"### Case `{opportunity_id}` — {error_type}",
        "",
        f"- **Thực tế:** {actual_text}.",
        (
            f"- **Model dự đoán:** {pred_text}, "
            f"với xác suất thắng "
            f"**{case['predicted_probability']:.3f}**."
        ),
        f"- **Loại lỗi:** `{error_type}`.",
        "",
        (
            "**Các feature có đóng góp lớn nhất "
            "vào prediction theo SHAP:**"
        ),
    ]

    # --------------------------------------------------------
    # Positive SHAP
    # --------------------------------------------------------

    lines += [
        "",
        "**Feature đẩy probability về phía WON:**"
    ]

    if len(positive) > 0:

        for r in positive.itertuples():

            lines.append(
                f"- `{r.feature}`: "
                f"SHAP = **{r.shap_value:+.4f}**"
            )

    else:

        lines.append(
            "- Không có feature trong Top-5 "
            "có SHAP dương."
        )

    # --------------------------------------------------------
    # Negative SHAP
    # --------------------------------------------------------

    lines += [
        "",
        "**Feature kéo probability khỏi phía WON:**"
    ]

    if len(negative) > 0:

        for r in negative.itertuples():

            lines.append(
                f"- `{r.feature}`: "
                f"SHAP = **{r.shap_value:+.4f}**"
            )

    else:

        lines.append(
            "- Không có feature trong Top-5 "
            "có SHAP âm."
        )

    # --------------------------------------------------------
    # Interpretation
    # --------------------------------------------------------

    if error_type == "FP":

        interpretation = (
            "Đây là trường hợp **False Positive (FP)**: "
            "model đánh giá Opportunity có khả năng WON "
            "cao nhưng thực tế Opportunity lại LOST. "
            "Các feature có SHAP dương là những tín hiệu "
            "khiến probability nghiêng về phía WON, "
            "nhưng các tín hiệu này không giúp model "
            "phân loại đúng outcome thực tế của case."
        )

    elif error_type == "FN":

        interpretation = (
            "Đây là trường hợp **False Negative (FN)**: "
            "model đánh giá probability WON thấp nhưng "
            "Opportunity thực tế lại WON. "
            "Các feature có SHAP âm là những tín hiệu "
            "kéo prediction khỏi phía WON, góp phần khiến "
            "model bỏ sót Opportunity này."
        )

    else:

        interpretation = (
            "Đây là trường hợp **borderline error**: "
            "probability nằm rất gần threshold 0.5, "
            "cho thấy model không có sự chắc chắn rõ ràng "
            "trong việc phân loại WON/LOST."
        )

    lines += [
        "",
        f"> **Diễn giải:** {interpretation}",
        "",
        (
            "> **Lưu ý:** SHAP mô tả mức đóng góp của "
            "feature đối với prediction của model, "
            "không chứng minh quan hệ nhân quả giữa "
            "feature và kết quả WON/LOST."
        ),
    ]

    return "\n".join(lines)


# ============================================================
# Hiển thị 5 Win error cases
# ============================================================

print("=" * 80)
print("LOCAL SHAP EXPLANATION — 5 WIN ERROR CASES")
print("=" * 80)


for case_no, (_, case) in enumerate(
    win_cases.iterrows(),
    start=1
):

    # --------------------------------------------------------
    # Tính SHAP
    # --------------------------------------------------------

    shap_table = catboost_local_shap_table(
        win_model,
        win_test,
        win_features,
        case.name,
    )

    # --------------------------------------------------------
    # Thông tin case
    # --------------------------------------------------------

    actual_text = (
        "WON"
        if int(case["label"]) == 1
        else "LOST"
    )

    predicted_text = (
        "WON"
        if int(case["predicted_label"]) == 1
        else "LOST"
    )

    print(
        f"\n{'-' * 80}\n"
        f"CASE {case_no}/5 | "
        f"Opportunity={case['opportunity_id']} | "
        f"Error={case['error_type']}\n"
        f"Thực tế={actual_text} | "
        f"Dự đoán={predicted_text} | "
        f"P(WIN)={case['predicted_probability']:.3f}"
    )

    # --------------------------------------------------------
    # Top-5 SHAP table
    # --------------------------------------------------------

    display(
        shap_table[
            [
                "feature",
                "shap_value",
                "abs_shap",
            ]
        ].head(5)
    )

    # --------------------------------------------------------
    # Markdown interpretation
    # --------------------------------------------------------

    display(
        Markdown(
            win_case_markdown(
                case,
                shap_table
            )
        )
    )

LOCAL SHAP EXPLANATION — 5 WIN ERROR CASES

--------------------------------------------------------------------------------
CASE 1/5 | Opportunity=CRM-OPP-2026-00148 | Error=FP
Thực tế=LOST | Dự đoán=WON | P(WIN)=0.934


,feature,shap_value,abs_shap
0,opportunity_amount,-0.564063,0.564063
1,lead_source,-0.519178,0.519178
2,days_lead_to_opp,0.295176,0.295176
3,customer_segment,0.200183,0.200183
4,discount_rate_at_first_quotation,0.037889,0.037889


### Case `CRM-OPP-2026-00148` — FP

- **Thực tế:** LOST.
- **Model dự đoán:** WON, với xác suất thắng **0.934**.
- **Loại lỗi:** `FP`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy probability về phía WON:**
- `days_lead_to_opp`: SHAP = **+0.2952**
- `customer_segment`: SHAP = **+0.2002**
- `discount_rate_at_first_quotation`: SHAP = **+0.0379**

**Feature kéo probability khỏi phía WON:**
- `opportunity_amount`: SHAP = **-0.5641**
- `lead_source`: SHAP = **-0.5192**

> **Diễn giải:** Đây là trường hợp **False Positive (FP)**: model đánh giá Opportunity có khả năng WON cao nhưng thực tế Opportunity lại LOST. Các feature có SHAP dương là những tín hiệu khiến probability nghiêng về phía WON, nhưng các tín hiệu này không giúp model phân loại đúng outcome thực tế của case.

> **Lưu ý:** SHAP mô tả mức đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và kết quả WON/LOST.


--------------------------------------------------------------------------------
CASE 2/5 | Opportunity=CRM-OPP-2026-00037 | Error=FP
Thực tế=LOST | Dự đoán=WON | P(WIN)=0.763


,feature,shap_value,abs_shap
0,territory,-0.865375,0.865375
1,customer_segment,-0.668345,0.668345
2,lead_source,0.464820,0.464820
3,opportunity_amount,-0.364640,0.364640
4,days_lead_to_opp,0.056474,0.056474


...

,feature,shap_value,abs_shap
0,customer_segment,0.593401,0.593401
1,opportunity_amount,0.410853,0.410853
2,territory,-0.201354,0.201354
3,days_lead_to_opp,0.139261,0.139261
4,lead_source,-0.097422,0.097422


### Case `CRM-OPP-2026-00007` — FN

- **Thực tế:** WON.
- **Model dự đoán:** LOST, với xác suất thắng **0.110**.
- **Loại lỗi:** `FN`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy probability về phía WON:**
- `customer_segment`: SHAP = **+0.5934**
- `opportunity_amount`: SHAP = **+0.4109**
- `days_lead_to_opp`: SHAP = **+0.1393**

**Feature kéo probability khỏi phía WON:**
- `territory`: SHAP = **-0.2014**
- `lead_source`: SHAP = **-0.0974**

> **Diễn giải:** Đây là trường hợp **False Negative (FN)**: model đánh giá probability WON thấp nhưng Opportunity thực tế lại WON. Các feature có SHAP âm là những tín hiệu kéo prediction khỏi phía WON, góp phần khiến model bỏ sót Opportunity này.

> **Lưu ý:** SHAP mô tả mức đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và kết quả WON/LOST.


--------------------------------------------------------------------------------
CASE 5/5 | Opportunity=CRM-OPP-2026-00034 | Error=FN
Thực tế=WON | Dự đoán=LOST | P(WIN)=0.455


,feature,shap_value,abs_shap
0,lead_source,-0.595763,0.595763
1,customer_segment,0.555975,0.555975
2,opportunity_amount,-0.135794,0.135794
3,days_lead_to_opp,0.014309,0.014309
4,territory,0.009789,0.009789


### Case `CRM-OPP-2026-00034` — FN

- **Thực tế:** WON.
- **Model dự đoán:** LOST, với xác suất thắng **0.455**.
- **Loại lỗi:** `FN`.

**Các feature có đóng góp lớn nhất vào prediction theo SHAP:**

**Feature đẩy probability về phía WON:**
- `customer_segment`: SHAP = **+0.5560**
- `days_lead_to_opp`: SHAP = **+0.0143**
- `territory`: SHAP = **+0.0098**

**Feature kéo probability khỏi phía WON:**
- `lead_source`: SHAP = **-0.5958**
- `opportunity_amount`: SHAP = **-0.1358**

> **Diễn giải:** Đây là trường hợp **False Negative (FN)**: model đánh giá probability WON thấp nhưng Opportunity thực tế lại WON. Các feature có SHAP âm là những tín hiệu kéo prediction khỏi phía WON, góp phần khiến model bỏ sót Opportunity này.

> **Lưu ý:** SHAP mô tả mức đóng góp của feature đối với prediction của model, không chứng minh quan hệ nhân quả giữa feature và kết quả WON/LOST.

# 5. Giới hạn dữ liệu và cách diễn giải

## 5.1. Historic-rate leakage qua `past_closed_all`

Trong pipeline sinh dữ liệu Delay, `past_closed_all` được dựng từ **toàn bộ bản ghi đã đóng trong hệ thống**, sau đó lọc `actual_close_date <= as_of` để tính historic rate. Cách dựng này được xác nhận trong script sinh dữ liệu. 

Sau khi đổi holdout sang random project split, historic pool không được cắt lại theo ranh giới train/test. Vì vậy, nếu một khách hàng hoặc PM xuất hiện ở cả hai phía split, historic-rate của một observation train/test có thể dùng các dự án thuộc phía còn lại.

Đây là **một giới hạn của feature engineering**, không phải lỗi của cách group split.

Theo audit đã được xác nhận trong quá trình rà soát dữ liệu:

- **14/20 khách hàng** có từ 2 dự án trở lên.

Vì vậy khả năng một customer xuất hiện ở cả hai phía split là có thật và cần được ghi nhận.

**Không dùng con số này để sửa lại metric DS-04 trong notebook; chỉ ghi như một limitation của pipeline.**

## 5.2. Win Probability

Tập train thực tế là **84 Opportunity**, test là **35 Opportunity**. Đây là số được đọc trực tiếp từ dataset, không hard-code trong phần phân tích.

Test ROC-AUC của CatBoost chỉ khoảng **0.513**, do đó không có cơ sở để diễn giải feature importance như những tín hiệu mạnh. DS-05 chủ động không trình bày Top-5 importance của Win và không tối ưu thêm AUC.

## 5.3. Kết luận về explainability

Explainability giúp mô tả **cách model đưa ra prediction**, không chứng minh quan hệ nhân quả.

Với Delay, MDI phải được đối chiếu với Permutation Importance và ablation trước khi kết luận về Top-5.

Với Win, do tín hiệu phân biệt trên test rất yếu, trọng tâm của DS-05 là **minh họa các lỗi prediction và giới hạn của model**, thay vì cố tạo ra các insight feature không được dữ liệu hỗ trợ.